# Example 06 · Multi-Agent: Subagents

**Source:** [LangChain Docs — Multi-Agent Subagents](https://docs.langchain.com/oss/python/langchain/multi-agent/subagents)

---

## Objectives

By the end of this notebook you will be able to:

1. Explain the **supervisor + subagent** architecture and when to use it
2. Implement the **Tool-Per-Agent** pattern — one wrapper tool per subagent
3. Implement the **Single Dispatch Tool** pattern — one `task()` tool that routes to any agent
4. Add **type-safe agent discovery** with an `Enum` constraint
5. Control **context flow** into and out of subagents

---

## Background

### The Problem with One Giant Agent

As tasks grow complex, a single agent's context window fills with history from all
sub-tasks — slowing it down and confusing its reasoning. The solution is **decomposition**:
split work across specialized agents, each with a clean slate.

### Supervisor Architecture

```
        User request
              │
              ▼
    ┌─────────────────┐
    │   Supervisor    │  ← main agent, coordinates work
    │   (orchestrator)│
    └────────┬────────┘
             │  calls subagents as tools
    ┌────────┼────────┐
    ▼        ▼        ▼
 research  writer  reviewer    ← each is a full agent
  agent    agent    agent
```

Key properties:
- **Centralized routing** — all decisions go through the supervisor
- **Context isolation** — each subagent call starts with a fresh context
- **Composable** — subagents can themselves have sub-subagents

### Two Invocation Patterns

| Pattern | Pros | Cons |
|---------|------|------|
| **Tool-Per-Agent** | Simple, full control per agent | Scales poorly with many agents |
| **Single Dispatch Tool** | One tool, N agents, easy to extend | Less per-agent customization |

Run cells **top to bottom**.

## Setup

In [ ]:
import sys
from pathlib import Path
from enum import Enum
from typing import Annotated

# 添加项目根目录，使 common/ 包可以被导入
_root = Path().resolve().parent.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from langchain.agents import create_agent
from langchain.tools import tool, InjectedToolCallId
from langchain_core.messages import ToolMessage
from langgraph.types import Command

from common.env import get_env   # loads .env
from common.llm import create_llm
from common.tracing import create_langfuse_handler, build_langfuse_config, get_langfuse_host

handler = create_langfuse_handler()

def make_config(trace_name: str, thread_id: str = "s06") -> dict:
    cfg = build_langfuse_config(handler, session_id="s06-subagents", trace_name=trace_name)
    cfg["configurable"] = {"thread_id": thread_id}
    return cfg

print("✓ Setup complete")

---

## Part 1 · Tool-Per-Agent Pattern

The simplest pattern: create each subagent independently, then wrap it in a
`@tool` function so the supervisor can call it like any other tool.

```
supervisor
  ├─ call_research_agent(query)  ←── @tool wrapping research_agent.invoke()
  └─ call_writer_agent(brief)   ←── @tool wrapping writer_agent.invoke()
```

### Step 1a — Create the subagents with their own tools

In [ ]:
# ── Tools for the research subagent ──────────────────────────────────────────
@tool
def fetch_facts(topic: str) -> str:
    """Fetch key facts about a topic from a knowledge base."""
    # 模拟知识库查询（实际项目中替换为真实 API 或 RAG 检索）
    facts = {
        "python": "Python was created by Guido van Rossum in 1991. It emphasizes readability.",
        "langchain": "LangChain is an open-source framework for building LLM-powered applications.",
        "agents": "AI agents combine LLMs with tools, enabling multi-step autonomous task execution.",
    }
    key = next((k for k in facts if k in topic.lower()), None)
    return facts[key] if key else f"No cached facts for '{topic}'. Reasoning from training data."

# ── Tools for the writer subagent ────────────────────────────────────────────
@tool
def count_words(text: str) -> str:
    """Count words in a draft and report if it meets a 50-word minimum."""
    n = len(text.split())
    status = "✓ meets minimum" if n >= 50 else f"✗ needs {50 - n} more words"
    return f"{n} words — {status}"

# ── Create the subagents ──────────────────────────────────────────────────────
research_agent = create_agent(
    model=create_llm(),
    tools=[fetch_facts],
    system_prompt=(
        "You are a research specialist. Use fetch_facts to gather information, "
        "then summarize your findings concisely."
    ),
)

writer_agent = create_agent(
    model=create_llm(),
    tools=[count_words],
    system_prompt=(
        "You are a writing specialist. Given research findings, "
        "write a clear, engaging explanation. Use count_words to verify length."
    ),
)

print("✓ Subagents created")

### Step 1b — Wrap subagents as tools

The wrapper function:
1. Formats the query into the subagent's expected input shape
2. Calls `subagent.invoke()`
3. Extracts and returns just the final message content — keeping the supervisor's context clean

In [ ]:
@tool("research", description="Research a topic and return a factual summary.")
def call_research_agent(query: str) -> str:
    """Delegate a research task to the research subagent."""
    result = research_agent.invoke(
        {"messages": [{"role": "user", "content": query}]},
        config=make_config("research-subagent", "s06-research"),
    )
    # create_agent returns GraphOutput — use .value to get the state dict
    return result["messages"][-1].content


@tool("write", description="Write a polished explanation given a research brief.")
def call_writer_agent(brief: str) -> str:
    """Delegate a writing task to the writer subagent."""
    result = writer_agent.invoke(
        {"messages": [{"role": "user", "content": brief}]},
        config=make_config("writer-subagent", "s06-writer"),
    )
    return result["messages"][-1].content


print("✓ Subagent tools registered")

### Step 1c — Create the supervisor and run

In [ ]:
supervisor = create_agent(
    model=create_llm(),
    tools=[call_research_agent, call_writer_agent],
    system_prompt=(
        "You are a content coordinator. "
        "First use 'research' to gather facts on a topic, "
        "then use 'write' to turn those facts into a polished explanation."
    ),
)

print("Running supervisor (Tool-Per-Agent)...\n" + "=" * 50)

for event in supervisor.stream(
    {"messages": [{"role": "user", "content": "Explain what LangChain agents are."}]},
    config=make_config("Tool-Per-Agent supervisor", "s06-p1"),
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

---

## Part 2 · Single Dispatch Tool Pattern

When you have many subagents, maintaining one tool per agent becomes unwieldy.
The **dispatch pattern** uses a single `task()` tool with two parameters:
- `agent_name` — which agent to call
- `description` — what to do

```
supervisor.task(agent_name="research", description="...")
supervisor.task(agent_name="writer",   description="...")
supervisor.task(agent_name="reviewer", description="...")
```

Adding a new agent only requires updating the registry — no new tool needed.

### Step 2a — Build the agent registry and dispatch tool

In [ ]:
# ── Tools for the reviewer subagent ──────────────────────────────────────────
@tool
def check_clarity(text: str) -> str:
    """Score text clarity: flag sentences longer than 25 words."""
    sentences = [s.strip() for s in text.replace('!','?').replace('.','?').split('?') if s.strip()]
    long_ones = [s for s in sentences if len(s.split()) > 25]
    if long_ones:
        return f"Found {len(long_ones)} overly long sentence(s). Consider splitting them."
    return "All sentences are concise. ✓"

reviewer_agent = create_agent(
    model=create_llm(),
    tools=[check_clarity],
    system_prompt=(
        "You are an editor. Review the given text for clarity and conciseness. "
        "Use check_clarity to identify issues, then provide actionable feedback."
    ),
)

# ── Central registry: agent_name → agent instance ────────────────────────────
AGENT_REGISTRY = {
    "research": research_agent,
    "writer":   writer_agent,
    "reviewer": reviewer_agent,
}

@tool
def task(agent_name: str, description: str) -> str:
    """Launch a subagent for a specific task.

    Available agents:
    - research: Gather facts and summarize a topic
    - writer:   Write a polished explanation from a brief
    - reviewer: Review text for clarity and suggest improvements
    """
    if agent_name not in AGENT_REGISTRY:
        return f"Unknown agent '{agent_name}'. Available: {list(AGENT_REGISTRY)}"
    agent = AGENT_REGISTRY[agent_name]
    result = agent.invoke(
        {"messages": [{"role": "user", "content": description}]},
        config=make_config(f"task-{agent_name}", f"s06-dispatch-{agent_name}"),
    )
    return result["messages"][-1].content

print("✓ Registry and dispatch tool ready")

### Step 2b — Supervisor using single dispatch

In [ ]:
dispatch_supervisor = create_agent(
    model=create_llm(),
    tools=[task],
    system_prompt=(
        "You are a content pipeline coordinator. Use the 'task' tool to delegate: "
        "first research the topic, then write an explanation, then review it. "
        "Available agents: research, writer, reviewer."
    ),
)

print("Running supervisor (Single Dispatch)...\n" + "=" * 50)

for event in dispatch_supervisor.stream(
    {"messages": [{"role": "user", "content": "Write and review a short explanation of Python."}]},
    config=make_config("Single Dispatch supervisor", "s06-p2"),
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

---

## Part 3 · Type-Safe Discovery with Enum

The string `agent_name` parameter in Part 2 is fragile — typos cause silent failures.
An **`Enum` constraint** makes the schema self-documenting and forces the LLM to pick
from a fixed set of valid names.

```python
class AgentName(str, Enum):
    RESEARCH = "research"
    WRITER   = "writer"
    REVIEWER = "reviewer"
```

The LLM sees the enum values in the tool's JSON schema and is constrained to choose one.

In [ ]:
# Enum constrains the supervisor to only valid agent names
class AgentName(str, Enum):
    RESEARCH = "research"
    WRITER   = "writer"
    REVIEWER = "reviewer"

@tool
def typed_task(agent_name: AgentName, description: str) -> str:
    """Launch a typed subagent. agent_name must be a valid AgentName value."""
    agent = AGENT_REGISTRY[agent_name.value]
    result = agent.invoke(
        {"messages": [{"role": "user", "content": description}]},
        config=make_config(f"typed-{agent_name.value}", f"s06-enum-{agent_name.value}"),
    )
    return result["messages"][-1].content

# Inspect the generated tool schema — the LLM sees this
import json
schema = typed_task.get_input_schema().model_json_schema()
print("Tool input schema (what the LLM sees):")
print(json.dumps(schema, indent=2))

In [ ]:
enum_supervisor = create_agent(
    model=create_llm(),
    tools=[typed_task],
    system_prompt="Coordinate subagents to research and write about any topic asked.",
)

print("Running supervisor (Enum-typed dispatch)...\n" + "=" * 50)

for event in enum_supervisor.stream(
    {"messages": [{"role": "user", "content": "Research and write about LangChain."}]},
    config=make_config("Enum-typed supervisor", "s06-p3"),
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

---

## Part 4 · Context Engineering

The default subagent tool receives only the string argument you pass it.
Sometimes you need to inject richer context — the supervisor's message history,
shared state values, or structured output back into the supervisor's state.

### Controlling Subagent Output with `Command`

By default the tool returns a `str`. With `Command` you can write directly to
the supervisor's state graph — useful when the subagent produces structured data
the supervisor should act on.

```
Supervisor state
  ├── messages: [...]
  └── research_notes: ""   ← subagent writes here via Command
```

In [ ]:
from langchain.agents import AgentState
from langgraph.types import Command

# 扩展 AgentState 以包含自定义字段
class SupervisorState(AgentState):
    research_notes: str   # subagent will populate this

@tool
def research_with_state(
    query: str,
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """Research a query and write findings into supervisor state research_notes."""
    result = research_agent.invoke(
        {"messages": [{"role": "user", "content": query}]},
        config=make_config("research-with-state", "s06-state"),
    )
    findings = result["messages"][-1].content

    # Command.update writes to the supervisor's state, not just returns a string
    return Command(update={
        "research_notes": findings,          # populate the custom field
        "messages": [ToolMessage(
            content=findings,
            tool_call_id=tool_call_id,
        )],
    })

state_supervisor = create_agent(
    model=create_llm(),
    tools=[research_with_state],
    state_schema=SupervisorState,
    system_prompt=(
        "Use research_with_state to gather facts. "
        "After researching, summarize what was found."
    ),
)

print("Running supervisor (Command output)...\n" + "=" * 50)

final = state_supervisor.invoke(
    {"messages": [{"role": "user", "content": "Research what Python is."}],
     "research_notes": ""},
    config=make_config("Command output supervisor", "s06-p4"),
)

print("\nFinal answer:")
final["messages"][-1].pretty_print()

print("\nresearch_notes field in state:")
print(final.get("research_notes", "(not set)"))

---

## Summary

| Pattern | API | Best for |
|---------|-----|----------|
| **Tool-Per-Agent** | One `@tool` per subagent | ≤5 agents, need custom input/output per agent |
| **Single Dispatch** | One `task(agent_name, description)` tool | Many agents, developed by different teams |
| **Enum constraint** | `agent_name: AgentName` in dispatch | Prevent typos, self-documenting schema |
| **Command output** | Return `Command(update={...})` from tool | Subagent writes structured data to supervisor state |

### Key Takeaways

1. **Subagents are tools** — the supervisor calls them exactly like any other tool
2. **Context isolation is by default** — each `subagent.invoke()` starts with a fresh conversation; use `checkpointer=True` only if you need continuations
3. **Return only what the supervisor needs** — extract `result["messages"][-1].content` to keep the supervisor's context lean
4. **Enum > string** for agent names — prevents hallucinated agent names and documents the registry in the schema
5. **`Command` for state writes** — use when the subagent produces structured data that should live in the supervisor's state, not just in the message history

In [ ]:
print(f"Traces available at: {get_langfuse_host()}")